In [ ]:
# clear all the parameters
dbutils.widgets.removeAll()

In [ ]:
%run "../utils/setup_databricks_session"

In [ ]:
from core.configs import WIDGETS_CONFIG
from core.engine import scoped_merge_engine
from core.helpers import process_sample_snapshot
from core.databricks_session import create_required_widgets, fetch_widget_values, log, view

In [ ]:
# define the required widgets
required_widgets = [
    "execution_mode",
    "silver_catalog",
    "silver_schema",
    "target_sample_fact",
    "target_sample_snapshot_fact",
    "run_daily",
    "run_weekly",
    "run_monthly",
]

# Create widgets for required parameters
create_required_widgets(dbutils, WIDGETS_CONFIG, required_widgets)

In [ ]:
# Get widget values
params = fetch_widget_values(dbutils, print_validation=False)

# Normalize run flags to boolean
run_daily_flag = str(params.run_daily).strip().lower() == "true"
run_weekly_flag = str(params.run_weekly).strip().lower() == "true"
run_monthly_flag = str(params.run_monthly).strip().lower() == "true"
log(f"Execution Mode: {params.execution_mode}")

In [ ]:
# get the required final append_df for the sample snapshot
sample_fact_df = spark.table(f"{params.silver_catalog}.{params.silver_schema}.{params.target_sample_fact}")
sample_snapshot_df, context = process_sample_snapshot(sample_fact_df, run_daily_flag, run_weekly_flag, run_monthly_flag)
view(sample_snapshot_df.count(), params.execution_mode)
view(sample_snapshot_df, params.execution_mode)

In [ ]:
# Determine if today falls within an active snapshot window
is_snapshot_window_active = (
    run_daily_flag
    or (run_weekly_flag and context.is_last_day_of_week)
    or (run_monthly_flag and context.is_last_day_of_month)
)
source = sample_snapshot_df
target = f"{params.silver_catalog}.{params.silver_schema}.{params.target_sample_snapshot_fact}"
merge_key = "sample_snapshot_record_source_cd"
write_mode = "append"
scope = f"t.sample_snapshot_date = '{context.snapshot_date}'"


# Commit only when within an active window
if is_snapshot_window_active:
    scoped_merge_engine(spark, source, target, merge_key, write_mode, scope)
else:
    print("No active snapshot window today. Skipping Delta write.")